# 예제 III · Response-Based Knowledge Distillation

사전 학습된 **ResNet-18 Teacher의 출력 분포**를 경량 ResNet-18 Student에 전달합니다. 비교할 두 Student는 같은 초기 가중치로 출발합니다. 한 쪽은 정답만, 다른 쪽은 정답과 Teacher의 Soft Target을 사용합니다.

[공개 Teacher 체크포인트](https://huggingface.co/edadaltocg/resnet18_cifar10)를 사용하며, 학습·평가는 CIFAR-10에서 수행합니다.

## 1. 데이터와 실행 환경

첫 실행에서 CIFAR-10을 준비합니다. 입력 정규화는 공개 Teacher의 CIFAR-10 설정과 맞췄습니다. 기본값은 **학습 128배치, 평가 512장**입니다.

두 Student가 같은 영상과 정답 배치를 받도록 Loader를 공유합니다. 평가에도 같은 테스트 샘플을 쓰므로 결과표의 모델 간 차이를 확인하기 쉽습니다.

**코드 셀 1 · 실행 장치와 KD 실습 설정**

GPU 또는 CPU를 선택하고 학습 128배치, 배치당 64장, 테스트 512장으로 비교 환경을 설정합니다.

In [ ]:
# 실습 설정을 한곳에서 관리한다. 데이터·가중치 캐시를 재사용하고 CUDA가 있으면 GPU로 실행한다.
import copy
import os
import time
from itertools import islice
from pathlib import Path
from statistics import mean

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# 동일한 시드와 배치 수를 사용해 반복 실행 조건을 맞춘다.
SEED = 50
BATCH_SIZE = 64
TRAIN_STEPS = 128
TEST_SAMPLES = 512
DATA_ROOT = Path(os.environ.get("AX_IDEC_DATA_ROOT", "./data"))
ASSET_ROOT = Path(os.environ.get("AX_IDEC_CACHE", "./idec_cache"))
ASSET_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("실행 장치:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU")

**코드 셀 2 · CIFAR-10 데이터 준비**

CIFAR-10에 RGB 정규화를 적용하고 두 Student가 공유할 학습 배치와 테스트 샘플을 고정합니다.

In [ ]:
# CIFAR-10 학습·테스트 분할에 같은 RGB 정규화를 적용한다. 테스트는 학습에 쓰이지 않는다.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
train_data = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=transform)

# 시드를 고정해 일부 샘플만 선택하고, 같은 데이터에서 반복 비교할 Loader를 만든다.
def make_loader(dataset, samples, seed):
    indices = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(seed))
    subset = Subset(dataset, indices[:samples].tolist())
    return DataLoader(subset, batch_size=BATCH_SIZE, num_workers=0,
                      pin_memory=(DEVICE.type == "cuda"))

train_loader = make_loader(train_data, (TRAIN_STEPS + 1) * BATCH_SIZE, SEED)
test_loader = make_loader(test_data, TEST_SAMPLES, SEED + 1)
print(f"학습 {TRAIN_STEPS} batches / 평가 {len(test_loader.dataset)} images")

## 2. Teacher 가중치 로드 및 Student 초기화

Teacher는 체크포인트에 맞는 ResNet-18 구조로 만든 뒤 평가 모드로 전환합니다. Teacher 파라미터는 고정하고, 학습 루프에서 Student만 업데이트합니다.

`student_ce`는 정답 학습 기준이고, `student_kd`는 그 초기 모델을 복제한 KD 모델입니다. 출력되는 Teacher 정확도는 현재 평가 샘플에서 체크포인트가 정상 동작하는지 확인하는 값입니다.

**코드 셀 3 · 경량 Student 구조 정의**

Depthwise Separable Convolution을 사용하는 Light ResNet-18을 두 학습 조건의 Student 구조로 정의합니다.

In [ ]:
# 기존 BasicBlock의 3×3 연산을 Depthwise와 Pointwise로 나누어 경량 모델을 만든다.
def build_light_resnet18():
    __all__ = ['ResNet', 'resnet18']

    def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
        """3x3 convolution with padding"""
        return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                         padding=dilation, groups=in_planes, bias=False, dilation=dilation)

    def conv1x1(in_planes, out_planes, stride=1):
        """1x1 convolution"""
        return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

    # 주 경로에서 특징을 계산하고 Residual 입력을 더한다. 크기가 바뀌는 블록에서는 Residual도 변환한다.
    class BasicBlock(nn.Module):
        expansion = 1

        def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                     base_width=64, dilation=1, norm_layer=None):
            super(BasicBlock, self).__init__()

            if norm_layer is None:
                norm_layer = nn.BatchNorm2d
            if groups != 1 or base_width != 64:
                raise ValueError(
                    'BasicBlock only supports groups=1 and base_width=64')
            if dilation > 1:
                raise NotImplementedError(
                    "Dilation > 1 not supported in BasicBlock")
            # Both self.conv1 and self.downsample layers downsample the input when stride != 1
            # 3×3 Depthwise에서 채널별 공간 특징을 계산하고, 1×1 Pointwise에서 채널을 조합한다.
            self.dwconv33_1 = conv3x3(inplanes, planes, stride)
            self.conv11_1 = conv1x1(planes, planes)
            self.bn1 = norm_layer(planes)
            self.relu = nn.ReLU(inplace=True)

            self.dwconv33_2 = conv3x3(planes, planes)
            self.conv11_2 = conv1x1(planes, planes)
            self.bn2 = norm_layer(planes)
            self.downsample = downsample
            self.stride = stride

        def forward(self, x):
            identity = x

            out = self.dwconv33_1(x)
            out = self.conv11_1(out)
            out = self.bn1(out)
            out = self.relu(out)

            out = self.dwconv33_2(out)
            out = self.conv11_2(out)
            out = self.bn2(out)

            if self.downsample is not None:
                identity = self.downsample(x)

            out += identity
            out = self.relu(out)

            return out

    # 입력 stem과 Residual stage를 연결하고, Global Average Pooling 뒤에서 클래스별
    # Logit을 계산한다.
    class ResNet(nn.Module):

        def __init__(self, block, layers, num_classes=1000, zero_init_residual=False,
                     groups=1, width_per_group=64, replace_stride_with_dilation=None,
                     norm_layer=None):
            super(ResNet, self).__init__()
            if norm_layer is None:
                norm_layer = nn.BatchNorm2d
            self._norm_layer = norm_layer

            self.inplanes = 64
            self.dilation = 1
            if replace_stride_with_dilation is None:
                # each element in the tuple indicates if we should replace
                # the 2x2 stride with a dilated convolution instead
                replace_stride_with_dilation = [False, False, False]
            if len(replace_stride_with_dilation) != 3:
                raise ValueError("replace_stride_with_dilation should be None "
                                 "or a 3-element tuple, got {}".format(replace_stride_with_dilation))
            self.groups = groups
            self.base_width = width_per_group
            self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3,
                                   bias=False)

            self.bn1 = norm_layer(self.inplanes)
            self.relu = nn.ReLU(inplace=True)
            self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
            # 네 stage는 64·128·256·512채널을 사용한다. 첫 블록의 stride로 해상도를 조정한다.
            self.layer1 = self._make_layer(block, 64, layers[0])
            self.layer2 = self._make_layer(block, 128, layers[1], stride=2,
                                           dilate=replace_stride_with_dilation[0])
            self.layer3 = self._make_layer(block, 256, layers[2], stride=2,
                                           dilate=replace_stride_with_dilation[1])
            self.layer4 = self._make_layer(block, 512, layers[3], stride=2,
                                           dilate=replace_stride_with_dilation[2])
            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
            self.fc = nn.Linear(512 * block.expansion, num_classes)

            # Convolution과 정규화 계층의 학습 초기값을 설정한다.
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(
                        m.weight, mode='fan_out', nonlinearity='relu')
                elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                    nn.init.constant_(m.weight, 1)
                    nn.init.constant_(m.bias, 0)

            # Zero-initialize the last BN in each residual branch,
            # so that the residual branch starts with zeros, and each residual block behaves like an identity.
            # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
            if zero_init_residual:
                for m in self.modules():
                    if isinstance(m, Bottleneck):
                        nn.init.constant_(m.bn3.weight, 0)
                    elif isinstance(m, BasicBlock):
                        nn.init.constant_(m.bn2.weight, 0)

        # 첫 블록에서 필요한 Residual 변환을 적용하고, 뒤의 블록은 같은 출력 크기로 이어 붙인다.
        def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
            norm_layer = self._norm_layer
            downsample = None
            previous_dilation = self.dilation
            if dilate:
                self.dilation *= stride
                stride = 1
            if stride != 1 or self.inplanes != planes * block.expansion:
                downsample = nn.Sequential(
                    conv1x1(self.inplanes, planes * block.expansion, stride),
                    norm_layer(planes * block.expansion),
                )

            layers = []
            layers.append(block(self.inplanes, planes, stride, downsample, self.groups,
                                self.base_width, previous_dilation, norm_layer))
            self.inplanes = planes * block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, planes, groups=self.groups,
                                    base_width=self.base_width, dilation=self.dilation,
                                    norm_layer=norm_layer))

            return nn.Sequential(*layers)

        # 입력에서 stem, 네 stage, Pooling, 최종 Linear Layer로 이어지는 순전파다.
        def _forward_impl(self, x):
            # See note [TorchScript super()]
            x = self.conv1(x)
            x = self.bn1(x)
            x = self.relu(x)
            x = self.maxpool(x)

            x = self.layer1(x)
            x = self.layer2(x)
            x = self.layer3(x)
            x = self.layer4(x)

            x = self.avgpool(x)
            x = torch.flatten(x, 1)
            x = self.fc(x)

            return x

        def forward(self, x):
            return self._forward_impl(x)

    # CIFAR-10에 맞게 첫 Conv2d를 3×3, stride 1로 설정하고 출력 클래스 수를 10으로 둔다.
    def resnet_18_cifar_lightweight():
        r"""
            Modify the resnet 18 network in order to run on cifar-10 dataset

            To enhance the accuracy, the (kernel_size, stride, padding) of conv1 is modified to (3, 1, 1)
            referenced by <https://github.com/akamaster/pytorch_resnet_cifar10>
        """
        model = ResNet(block=BasicBlock, layers=[2, 2, 2, 2], num_classes=10)

        model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1,
                                bias=False)

        return model

    return resnet_18_cifar_lightweight()

**코드 셀 4 · 학습과 평가 함수 정의**

분류 정확도, MACs와 파라미터 수, Cross-Entropy 학습을 계산하는 공통 함수를 준비합니다.

In [ ]:
# 평가 모드에서 각 영상의 최댓값 클래스를 정답과 비교해 전체 정확도를 집계한다.
@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        correct += (model(images).argmax(1) == labels).sum().item()
        total += labels.numel()
    return 100 * correct / total

# Conv2d와 Linear에 Hook을 붙여 이미지 한 장의 Dense MACs를 구하고 전체 파라미터 수를 센다.
def macs_and_parameters(model, input_shape=(3, 32, 32)):
    # Conv2d와 Linear의 multiply-accumulate만 집계.
    operations = 0
    def count(module, args, output):
        nonlocal operations
        if isinstance(module, nn.Conv2d):
            kernel = module.kernel_size[0] * module.kernel_size[1]
            operations += output.numel() * (module.in_channels // module.groups) * kernel
        else:
            operations += output.numel() * module.in_features
    # 실행 중 필요한 계층에만 Hook을 연결하고 블록이 끝나면 제거한다.
    hooks = [m.register_forward_hook(count) for m in model.modules()
             if isinstance(m, (nn.Conv2d, nn.Linear))]
    previous_mode = model.training
    try:
        model.eval()
        with torch.inference_mode():
            model(torch.zeros(1, *input_shape, device=DEVICE))
    finally:
        for hook in hooks:
            hook.remove()
        model.train(previous_mode)
    return operations, sum(p.numel() for p in model.parameters())

# 정해진 배치만 처리하며, Cross-Entropy의 gradient로 모델을 갱신한다.
def train_steps(model, loader, steps):
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    losses = []
    start = time.perf_counter()
    # 학습 배치를 순회하며 gradient 초기화 → Forward → 역전파 → 갱신을 반복한다.
    for images, labels in islice(loader, steps):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(images), labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses, time.perf_counter() - start

**코드 셀 5 · Teacher와 두 Student 준비**

사전 학습된 ResNet-18 Teacher를 고정하고 동일한 가중치로 시작하는 두 Student를 만든 뒤 Teacher의 테스트 정확도를 출력합니다.

In [ ]:
from torchvision.models import resnet18

# CIFAR-10으로 사전 학습한 공개 Teacher의 체크포인트를 불러온다.
TEACHER_URL = ("https://huggingface.co/edadaltocg/resnet18_cifar10/resolve/"
               "ccacbae365101b9bfa45e1f17f4acac30c2601b2/pytorch_model.bin")
teacher = resnet18(weights=None, num_classes=10)
teacher.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
teacher.maxpool = nn.Identity()
state_dict = torch.hub.load_state_dict_from_url(
    TEACHER_URL, model_dir=str(ASSET_ROOT), file_name="teacher.bin",
    map_location="cpu", weights_only=True
)
teacher.load_state_dict(state_dict, strict=True)
# Offline KD에서는 Teacher를 평가 모드에 두고 가중치를 고정한다.
teacher = teacher.to(DEVICE).eval().requires_grad_(False)
student_ce = build_light_resnet18().to(DEVICE)
student_kd = copy.deepcopy(student_ce)
print(f"Teacher 정확도 ({len(test_loader.dataset)}장): {evaluate(teacher, test_loader):.2f}%")

## 3. Soft Target과 KD Loss

**Temperature T**는 Softmax에 적용하는 양수입니다. Softmax를 계산하기 전에 각 클래스의 Logit $z_i$를 **T로 나눠** $z_i/T$를 입력으로 사용합니다. 한 영상에서 **T=1**과 **T=4**의 상위 세 클래스 확률을 비교해 확률이 얼마나 퍼지는지 확인합니다.

`response_kd()`는 같은 T를 적용한 Teacher 확률과 Student 로그 확률 사이의 KL divergence를 계산합니다. 코드에서 **Teacher와 Student에 모두 T를 적용하는 줄**과 **손실에 T²를 곱하는 줄**을 찾습니다.

**코드 셀 6 · Temperature와 Response KD Loss**

KL Divergence로 Response KD Loss를 정의하고 같은 이미지 한 장에서 Teacher의 T=1과 T=4에 따른 상위 세 클래스의 확률을 비교합니다.

In [ ]:
# 모든 Logit을 같은 양수 T로 나눈 뒤 Softmax를 적용한다. 함께 둔 계수는 KD 손실의 비중을 정한다.
TEMPERATURE = 4.0
KD_WEIGHT = 0.5
KD_STEPS = TRAIN_STEPS

def response_kd(student_logits, teacher_logits, temperature):
    student_log_prob = F.log_softmax(student_logits / temperature, dim=1)
    teacher_prob = F.softmax(teacher_logits / temperature, dim=1)
    return F.kl_div(student_log_prob, teacher_prob, reduction="batchmean") * temperature**2

images, _ = next(iter(test_loader))
with torch.inference_mode():
    teacher_logits = teacher(images[:1].to(DEVICE))
    # 같은 영상의 Logit을 사용해 T만 바꿨을 때 상위 확률이 어떻게 달라지는지 비교한다.
    for temperature in (1.0, TEMPERATURE):
        probs, indices = (teacher_logits / temperature).softmax(1).topk(3)
        print(f"T={temperature:g}:", [(test_data.classes[i], round(p, 3))
                                    for p, i in zip(probs[0].tolist(), indices[0].tolist())])

## 4. 정답만 학습한 Student와 KD Student 비교

- **정답만 학습** · 실제 클래스에 대한 Cross-Entropy 사용
- **Response-Based KD** · 실제 클래스에 대한 Cross-Entropy와 Teacher 출력 분포에 대한 KD Loss 사용

마지막 배치의 손실과 두 Student의 테스트 정확도를 확인합니다. 이번 비교는 제한된 학습 배치와 테스트 영상에서 얻은 값입니다. 한 번의 짧은 실행에서는 KD를 적용한 정답률이 더 낮거나 비슷할 수도 있습니다.

**코드 셀 7 · 정답만 학습한 Student와 KD Student 비교**

같은 128배치에서 두 Student를 각각 학습하고 마지막 배치의 Cross-Entropy와 KD Loss 및 테스트 512장의 정확도를 출력합니다.

In [ ]:
optim_ce = torch.optim.SGD(student_ce.parameters(), lr=0.03, momentum=0.9)
optim_kd = torch.optim.SGD(student_kd.parameters(), lr=0.03, momentum=0.9)
history = []
start = time.perf_counter()
for images, labels in islice(train_loader, KD_STEPS):
    images, labels = images.to(DEVICE), labels.to(DEVICE)
    # Teacher는 학습 목표를 만드는 데 사용한다. 이 Forward에서는 Teacher의 gradient를 계산하지
    # 않는다.
    with torch.no_grad():
        targets = teacher(images)
    optim_ce.zero_grad(set_to_none=True)
    loss_ce = F.cross_entropy(student_ce(images), labels)
    loss_ce.backward()
    optim_ce.step()

    # 같은 배치에서 KD Student에는 Teacher의 전달 신호를 추가한다.
    optim_kd.zero_grad(set_to_none=True)
    logits = student_kd(images)
    loss_label = F.cross_entropy(logits, labels)
    loss_kd = response_kd(logits, targets, TEMPERATURE)
    loss = (1 - KD_WEIGHT) * loss_label + KD_WEIGHT * loss_kd
    loss.backward()
    optim_kd.step()
    history.append((loss_ce.item(), loss_label.item(), loss_kd.item()))

last = history[-1]
print(f"{len(history)} batches · {time.perf_counter() - start:.1f} s")
print(f"마지막 배치: 기본 CE {last[0]:.3f} | KD 모델 CE {last[1]:.3f} | KD Loss {last[2]:.3f}")
print(f"정답만 학습: {evaluate(student_ce, test_loader):.2f}%")
print(f"정답 + 증류: {evaluate(student_kd, test_loader):.2f}%")